# Timestamp Distribution Analysis
**Cases vs Counterfactuals — Time Elapsed from Admission (t₀)**

For each matched pair (rank=1), time zero is the encounter admission time (`EN_START_DATE + EN_START_TIME`).  
Each clinical event is re-expressed as **hours elapsed since admission**.  
Distributions are compared between **PSI cases** (events that developed a complication) and their **matched counterfactuals**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

ROOT = Path('../')
RAW  = ROOT / 'data/raw/psi_tables'

sns.set_theme(style='whitegrid', font_scale=1.05)
CASE_COLOR  = '#e05c4b'   # coral-red for cases
CTRL_COLOR  = '#4b8ec8'   # steel-blue for counterfactuals
ALPHA       = 0.45
BW_ADJUST   = 0.8

print('Setup complete.')

## 1. Load Matched Pairs and Build t₀ Lookup

In [ ]:
# Load rank-1 matched pairs (primary counterfactual per case)
pairs = pd.read_csv(ROOT / 'results/tables/all_matched_pairs.csv')
pairs_r1 = pairs[pairs['match_rank'] == 1].copy()

case_encs  = set(pairs_r1['case_enc'])
donor_encs = set(pairs_r1['donor_enc'])
all_encs   = case_encs | donor_encs

print(f'Matched pairs (rank=1): {len(pairs_r1)}')
print(f'  Cases:           {len(case_encs)}')
print(f'  Counterfactuals: {len(donor_encs)}')
print(f'  PSI types:       {pairs_r1["psi_type"].nunique()}')

In [ ]:
def parse_dt(date_series, time_series, default_time='12:00:00'):
    """Combine date + time columns into a single datetime series."""
    time_filled = time_series.fillna(default_time).astype(str)
    combined = date_series.astype(str) + ' ' + time_filled
    return pd.to_datetime(combined, errors='coerce')


def elapsed_hours(event_ts, t0_map, enc_col):
    """Return hours elapsed since admission for each event row."""
    t0 = enc_col.map(t0_map)
    return (event_ts - t0).dt.total_seconds() / 3600


# Load encounters for t0
enc = pd.read_csv(RAW / 'encounters.csv', usecols=['ENCOUNTER_ID', 'EN_START_DATE', 'EN_START_TIME', 'EN_LOS'])
enc = enc[enc['ENCOUNTER_ID'].isin(all_encs)].copy()
enc['t0'] = parse_dt(enc['EN_START_DATE'], enc['EN_START_TIME'])

t0_map  = enc.set_index('ENCOUNTER_ID')['t0'].to_dict()
los_map = enc.set_index('ENCOUNTER_ID')['EN_LOS'].to_dict()

# Role labels for all encounters
role = {enc_id: 'Case' for enc_id in case_encs}
role.update({enc_id: 'Counterfactual' for enc_id in donor_encs})

print(f'Encounters with t0 resolved: {enc["t0"].notna().sum()} / {len(enc)}')

## 2. Helper: Plot Distribution per Domain

In [ ]:
def plot_elapsed_distribution(
    df_elapsed,          # DataFrame with columns: elapsed_h, role
    title,
    xlabel='Hours from admission',
    xlim=(-12, 240),     # default: -12h to 10 days
    ax=None,
    bins=60,
):
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4))

    palette = {'Case': CASE_COLOR, 'Counterfactual': CTRL_COLOR}

    for role_label, color in palette.items():
        data = df_elapsed.loc[df_elapsed['role'] == role_label, 'elapsed_h'].dropna()
        data = data[(data >= xlim[0]) & (data <= xlim[1])]
        if data.empty:
            continue
        ax.hist(data, bins=bins, range=xlim, alpha=ALPHA, color=color,
                label=f'{role_label} (n={len(data):,})', density=True)
        data.plot.kde(ax=ax, color=color, linewidth=2, bw_method=BW_ADJUST,
                      label='_nolegend_')

    ax.set_title(title, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_xlim(xlim)
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.6, label='Admission (t₀)')
    ax.legend(fontsize=9)
    return ax


print('Plot helper defined.')

## 3. Domain Analysis

### 3a. Diagnoses

In [ ]:
dx = pd.read_csv(RAW / 'diagnoses.csv', usecols=['ENCOUNTER_ID', 'DX_DATE', 'DX_TIME', 'DX_CODE', 'DX_HCS_DESC'])
dx = dx[dx['ENCOUNTER_ID'].isin(all_encs)].copy()
dx['event_ts']  = parse_dt(dx['DX_DATE'], dx['DX_TIME'])
dx['elapsed_h'] = elapsed_hours(dx['event_ts'], t0_map, dx['ENCOUNTER_ID'])
dx['role']      = dx['ENCOUNTER_ID'].map(role)

print(f'Diagnoses rows: {len(dx):,}  |  valid elapsed: {dx["elapsed_h"].notna().sum():,}')
dx[['ENCOUNTER_ID', 'DX_CODE', 'DX_HCS_DESC', 'elapsed_h', 'role']].head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_elapsed_distribution(dx[['elapsed_h', 'role']], title='Diagnoses — Time from Admission', xlim=(-24, 480), ax=ax)
plt.tight_layout()
plt.show()

### 3b. Laboratory Orders

In [ ]:
labs = pd.read_csv(RAW / 'labs.csv', usecols=[
    'ENCOUNTER_ID', 'LB_ORDER_DATE', 'LB_ORDER_TIME',
    'LB_SPECIMEN_DATE', 'LB_SPECIMEN_TIME',
    'LB_RESULT_DATE', 'LB_RESULT_TIME',
    'LB_SHORT_NAME', 'LB_LOINC_LEVEL2_CAT'
])
labs = labs[labs['ENCOUNTER_ID'].isin(all_encs)].copy()

# Use specimen time (most clinically meaningful); fall back to order time
labs['specimen_ts'] = parse_dt(labs['LB_SPECIMEN_DATE'], labs['LB_SPECIMEN_TIME'])
labs['order_ts']    = parse_dt(labs['LB_ORDER_DATE'],    labs['LB_ORDER_TIME'])
labs['result_ts']   = parse_dt(labs['LB_RESULT_DATE'],   labs['LB_RESULT_TIME'])
labs['event_ts']    = labs['specimen_ts'].fillna(labs['order_ts'])

labs['elapsed_h']  = elapsed_hours(labs['event_ts'], t0_map, labs['ENCOUNTER_ID'])
labs['result_lag'] = (labs['result_ts'] - labs['event_ts']).dt.total_seconds() / 3600
labs['role']       = labs['ENCOUNTER_ID'].map(role)

print(f'Lab rows: {len(labs):,}  |  valid elapsed: {labs["elapsed_h"].notna().sum():,}')
print('Top lab categories:', labs['LB_LOINC_LEVEL2_CAT'].value_counts().head(5).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_elapsed_distribution(
    labs[['elapsed_h', 'role']],
    title='Labs — Specimen/Order Time from Admission',
    xlim=(-12, 240), ax=axes[0]
)

# Result turnaround time (specimen → result)
result_lag_df = labs[['result_lag', 'role']].rename(columns={'result_lag': 'elapsed_h'})
plot_elapsed_distribution(
    result_lag_df,
    title='Labs — Result Turnaround Time (Specimen → Result)',
    xlabel='Hours (specimen to result)',
    xlim=(0, 48), ax=axes[1]
)
axes[1].get_lines()[0].set_label('_nolegend_')  # remove t0 label from turnaround plot
axes[1].lines[0].set_visible(False)             # hide t0 vline (not meaningful here)

plt.tight_layout()
plt.show()

In [ ]:
# Breakdown by lab category
top_cats = labs['LB_LOINC_LEVEL2_CAT'].value_counts().head(6).index.tolist()
ncols = 3
nrows = (len(top_cats) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
axes = axes.flatten()

for i, cat in enumerate(top_cats):
    subset = labs[labs['LB_LOINC_LEVEL2_CAT'] == cat][['elapsed_h', 'role']]
    plot_elapsed_distribution(subset, title=f'Labs: {cat}', xlim=(-12, 240), ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Lab Distributions by Category', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3c. Vitals

In [ ]:
vs = pd.read_csv(RAW / 'vitals.csv', usecols=['ENCOUNTER_ID', 'VS_DATE', 'VS_TIME', 'VS_CODE', 'VS_DESC'])
vs = vs[vs['ENCOUNTER_ID'].isin(all_encs)].copy()
vs['event_ts']  = parse_dt(vs['VS_DATE'], vs['VS_TIME'])
vs['elapsed_h'] = elapsed_hours(vs['event_ts'], t0_map, vs['ENCOUNTER_ID'])
vs['role']      = vs['ENCOUNTER_ID'].map(role)

print(f'Vitals rows: {len(vs):,}  |  valid elapsed: {vs["elapsed_h"].notna().sum():,}')
print('Vital types:', vs['VS_DESC'].value_counts().head(8).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_elapsed_distribution(vs[['elapsed_h', 'role']], title='Vitals — Time from Admission', xlim=(-12, 240), ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Breakdown by vital sign type
top_vs = vs['VS_DESC'].value_counts().head(6).index.tolist()
ncols = 3
nrows = (len(top_vs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
axes = axes.flatten()

for i, vtype in enumerate(top_vs):
    subset = vs[vs['VS_DESC'] == vtype][['elapsed_h', 'role']]
    plot_elapsed_distribution(subset, title=f'Vital: {vtype}', xlim=(-12, 240), ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Vital Sign Distributions by Type', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3d. Procedures

In [ ]:
px = pd.read_csv(RAW / 'procedures.csv', usecols=[
    'ENCOUNTER_ID', 'PX_ORDER_DATE', 'PX_ORDER_TIME',
    'PX_SERVICE_DATE', 'PX_SERVICE_TIME', 'PX_CODE', 'PX_SHORT_DESC', 'PX_TYPE'
])
px = px[px['ENCOUNTER_ID'].isin(all_encs)].copy()

# Service date is the actual procedure time; fall back to order date
px['service_ts'] = parse_dt(px['PX_SERVICE_DATE'], px['PX_SERVICE_TIME'])
px['order_ts']   = parse_dt(px['PX_ORDER_DATE'],   px['PX_ORDER_TIME'])
px['event_ts']   = px['service_ts'].fillna(px['order_ts'])

px['elapsed_h']  = elapsed_hours(px['event_ts'], t0_map, px['ENCOUNTER_ID'])
px['order_lag']  = (px['service_ts'] - px['order_ts']).dt.total_seconds() / 3600
px['role']       = px['ENCOUNTER_ID'].map(role)

print(f'Procedure rows: {len(px):,}  |  valid elapsed: {px["elapsed_h"].notna().sum():,}')
print('Top procedure types:', px['PX_TYPE'].value_counts().head(5).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_elapsed_distribution(
    px[['elapsed_h', 'role']],
    title='Procedures — Service Time from Admission',
    xlim=(-24, 480), ax=axes[0]
)

# Order-to-service lag
lag_df = px[px['order_lag'].between(0, 72)][['order_lag', 'role']].rename(columns={'order_lag': 'elapsed_h'})
plot_elapsed_distribution(
    lag_df,
    title='Procedures — Order-to-Service Lag',
    xlabel='Hours (order to service)',
    xlim=(0, 72), ax=axes[1]
)
axes[1].lines[0].set_visible(False)  # hide t0 vline

plt.tight_layout()
plt.show()

### 3e. Prescription Orders

In [ ]:
rx = pd.read_csv(RAW / 'prescription_orders.csv', usecols=[
    'ENCOUNTER_ID', 'RX_ORDER_DATE', 'RX_ORDER_TIME',
    'RX_START_DATE', 'RX_START_TIME', 'RX_END_DATE', 'RX_END_TIME',
    'RX_GENERIC_NAME', 'RX_ORDER_CATEGORY', 'RX_STATUS'
])
rx = rx[rx['ENCOUNTER_ID'].isin(all_encs)].copy()

rx['order_ts'] = parse_dt(rx['RX_ORDER_DATE'], rx['RX_ORDER_TIME'])
rx['start_ts'] = parse_dt(rx['RX_START_DATE'], rx['RX_START_TIME'])
rx['end_ts']   = parse_dt(rx['RX_END_DATE'],   rx['RX_END_TIME'])
rx['event_ts'] = rx['order_ts']

rx['elapsed_h']    = elapsed_hours(rx['event_ts'], t0_map, rx['ENCOUNTER_ID'])
rx['duration_h']   = (rx['end_ts'] - rx['start_ts']).dt.total_seconds() / 3600
rx['role']         = rx['ENCOUNTER_ID'].map(role)

print(f'Rx order rows: {len(rx):,}  |  valid elapsed: {rx["elapsed_h"].notna().sum():,}')
print('Top order categories:', rx['RX_ORDER_CATEGORY'].value_counts().head(5).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_elapsed_distribution(
    rx[['elapsed_h', 'role']],
    title='Prescription Orders — Time from Admission',
    xlim=(-12, 240), ax=axes[0]
)

# Order duration
dur_df = rx[rx['duration_h'].between(0, 240)][['duration_h', 'role']].rename(columns={'duration_h': 'elapsed_h'})
plot_elapsed_distribution(
    dur_df,
    title='Prescription Orders — Order Duration',
    xlabel='Duration (hours)',
    xlim=(0, 240), ax=axes[1]
)
axes[1].lines[0].set_visible(False)

plt.tight_layout()
plt.show()

### 3f. Prescription Administrations

In [ ]:
adm = pd.read_csv(RAW / 'prescription_administrations.csv', usecols=[
    'ENCOUNTER_ID', 'AD_ADMIN_DATE', 'AD_ADMIN_TIME',
    'AD_GENERIC_NAME', 'AD_HCS_ADMIN_ROUTE'
])
adm = adm[adm['ENCOUNTER_ID'].isin(all_encs)].copy()
adm['event_ts']  = parse_dt(adm['AD_ADMIN_DATE'], adm['AD_ADMIN_TIME'])
adm['elapsed_h'] = elapsed_hours(adm['event_ts'], t0_map, adm['ENCOUNTER_ID'])
adm['role']      = adm['ENCOUNTER_ID'].map(role)

print(f'Rx admin rows: {len(adm):,}  |  valid elapsed: {adm["elapsed_h"].notna().sum():,}')
print('Top admin routes:', adm['AD_HCS_ADMIN_ROUTE'].value_counts().head(5).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_elapsed_distribution(
    adm[['elapsed_h', 'role']],
    title='Medication Administrations — Time from Admission',
    xlim=(-12, 240), ax=ax
)
plt.tight_layout()
plt.show()

### 3g. Clinical Scores

In [ ]:
scores = pd.read_csv(RAW / 'scores.csv', usecols=[
    'ENCOUNTER_ID', 'QS_DATE', 'QS_TIME', 'QS_NAME', 'QS_MEASURE_NAME'
])
scores = scores[scores['ENCOUNTER_ID'].isin(all_encs)].copy()
scores['event_ts']  = parse_dt(scores['QS_DATE'], scores['QS_TIME'])
scores['elapsed_h'] = elapsed_hours(scores['event_ts'], t0_map, scores['ENCOUNTER_ID'])
scores['role']      = scores['ENCOUNTER_ID'].map(role)

print(f'Score rows: {len(scores):,}  |  valid elapsed: {scores["elapsed_h"].notna().sum():,}')
print('Top score types:', scores['QS_NAME'].value_counts().head(6).to_dict())

In [ ]:
if len(scores) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_elapsed_distribution(
        scores[['elapsed_h', 'role']],
        title='Clinical Scores — Time from Admission',
        xlim=(-12, 240), ax=ax
    )
    plt.tight_layout()
    plt.show()
else:
    print('No clinical scores for the matched encounters.')

### 3h. Medical Devices

In [ ]:
devs = pd.read_csv(RAW / 'medical_devices.csv', usecols=[
    'ENCOUNTER_ID', 'DV_IMPLANT_DATE', 'DV_REMOVAL_DATE',
    'DV_DEVICE_TYPE', 'DV_HCS_DEVICE_NAME'
])
devs = devs[devs['ENCOUNTER_ID'].isin(all_encs)].copy()
devs['implant_ts']   = pd.to_datetime(devs['DV_IMPLANT_DATE'], errors='coerce')
devs['removal_ts']   = pd.to_datetime(devs['DV_REMOVAL_DATE'], errors='coerce')
devs['elapsed_h']    = elapsed_hours(devs['implant_ts'], t0_map, devs['ENCOUNTER_ID'])
devs['duration_h']   = (devs['removal_ts'] - devs['implant_ts']).dt.total_seconds() / 3600
devs['role']         = devs['ENCOUNTER_ID'].map(role)

print(f'Device rows: {len(devs):,}  |  valid elapsed: {devs["elapsed_h"].notna().sum():,}')
if len(devs) > 0:
    print('Top device types:', devs['DV_DEVICE_TYPE'].value_counts().head(5).to_dict())

In [ ]:
if devs['elapsed_h'].notna().sum() > 10:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_elapsed_distribution(
        devs[['elapsed_h', 'role']],
        title='Medical Device Implants — Time from Admission',
        xlim=(-24, 480), ax=ax
    )
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient device implant events for distribution plot.')

## 4. Summary Dashboard — All Domains

In [ ]:
domains = [
    ('Diagnoses',             dx,  (-24, 480)),
    ('Lab Orders',            labs, (-12, 240)),
    ('Vitals',                vs,  (-12, 240)),
    ('Procedures',            px,  (-24, 480)),
    ('Rx Orders',             rx,  (-12, 240)),
    ('Rx Administrations',    adm, (-12, 240)),
    ('Clinical Scores',       scores, (-12, 240)),
]

# Filter out empty domains
domains = [(name, df, xlim) for name, df, xlim in domains if df['elapsed_h'].notna().sum() > 5]

ncols = 2
nrows = (len(domains) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for i, (name, df, xlim) in enumerate(domains):
    plot_elapsed_distribution(
        df[['elapsed_h', 'role']],
        title=name,
        xlim=xlim,
        ax=axes[i],
        bins=50
    )

for j in range(len(domains), len(axes)):
    axes[j].set_visible(False)

# Shared legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor=CASE_COLOR, alpha=0.6, label='Case'),
    Patch(facecolor=CTRL_COLOR, alpha=0.6, label='Counterfactual'),
    Line2D([0], [0], color='black', linestyle='--', linewidth=1.5, label='Admission (t₀)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, -0.02), frameon=True)

fig.suptitle(
    'Clinical Event Timing — Cases vs Counterfactuals\n(Hours from Admission)',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(ROOT / 'results/figures/timestamp_distributions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/figures/timestamp_distributions.png')

## 5. Early Window Focus (First 4 Hours)

This mirrors the feature extraction window used in the propensity model — events within [t₀, t₀+4h].

In [ ]:
EARLY_WINDOW = 4  # hours

early_domains = [
    ('Diagnoses',          dx),
    ('Lab Orders',         labs),
    ('Vitals',             vs),
    ('Procedures',         px),
    ('Rx Orders',          rx),
    ('Rx Administrations', adm),
]

ncols = 3
nrows = (len(early_domains) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for i, (name, df) in enumerate(early_domains):
    subset = df[df['elapsed_h'].between(0, EARLY_WINDOW)][['elapsed_h', 'role']]
    n_case = (subset['role'] == 'Case').sum()
    n_ctrl = (subset['role'] == 'Counterfactual').sum()
    if len(subset) < 5:
        axes[i].set_title(f'{name}\n(insufficient events in first 4h)', fontsize=10)
        axes[i].axis('off')
        continue
    plot_elapsed_distribution(
        subset,
        title=f'{name}\n(cases: {n_case}, controls: {n_ctrl})',
        xlabel='Hours from admission',
        xlim=(0, EARLY_WINDOW),
        bins=24,
        ax=axes[i]
    )
    axes[i].lines[0].set_visible(False)  # hide t0 vline inside window

for j in range(len(early_domains), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f'Clinical Events in First {EARLY_WINDOW}h of Admission (Feature Extraction Window)',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 6. Event Count Summary Table

In [ ]:
summary_rows = []
all_domains = [
    ('Diagnoses',          dx),
    ('Labs',               labs),
    ('Vitals',             vs),
    ('Procedures',         px),
    ('Rx Orders',          rx),
    ('Rx Administrations', adm),
    ('Clinical Scores',    scores),
    ('Medical Devices',    devs),
]

for name, df in all_domains:
    valid = df[df['elapsed_h'].notna()]
    for role_label in ['Case', 'Counterfactual']:
        sub = valid[valid['role'] == role_label]['elapsed_h']
        early = sub[sub.between(0, 4)]
        summary_rows.append({
            'Domain':          name,
            'Group':           role_label,
            'N events':        len(sub),
            'N events (0-4h)': len(early),
            'Median (h)':      round(sub.median(), 1) if len(sub) else None,
            'P25 (h)':         round(sub.quantile(0.25), 1) if len(sub) else None,
            'P75 (h)':         round(sub.quantile(0.75), 1) if len(sub) else None,
        })

summary = pd.DataFrame(summary_rows)
summary